# 05 — Weekly Injury Adjustment

This notebook adds a small, player sensitive injury layer to the weekly model.

The goal is not to treat every injury as a major team strength event. NFL rosters contain replacement level talent, and the next player up can often absorb a meaningful portion of the missing player's role.

The adjustment therefore depends on:

1. **Expected availability** from the current injury/practice report.
2. **Player role** from the latest depth chart.
3. **Position importance**, with quarterback handled separately.
4. **Replacement quality**, represented conservatively by depth chart structure.
5. **A hard cap** so injuries cannot overwhelm the underlying team strength model.

This is intentionally a secondary adjustment. Preseason strength and observed team performance remain the primary drivers.

The original preseason notebooks and ratings are never modified.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import nflreadpy as nfl


In [2]:
PROJECT_ROOT = Path("../..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 12

TEAM_STRENGTH_PATH = (
    PROCESSED_DIR / "weekly"
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)

OUTPUT_PATH = (
    WEEKLY_DATA_DIR / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

INJURY_DETAIL_PATH = (
    WEEKLY_DATA_DIR / f"week_{TARGET_WEEK:02d}_injury_adjustments.parquet"
)


## Load Current Injury Reports and Depth Charts

`nflreadpy` is used so this can be rerun as the official injury report changes during the week.

**Important:** early in the week, official game status designations may not yet be available. In that case, the notebook uses practice participation conservatively and prints a warning. Rerun this notebook closer to kickoff as reports become more complete.


In [3]:

# Load Week 9 team strengths and current player availability

team_strength = pd.read_parquet(TEAM_STRENGTH_PATH)

injuries = nfl.load_injuries([SEASON]).to_pandas()
depth = nfl.load_depth_charts([SEASON]).to_pandas()
rosters = nfl.load_rosters([SEASON]).to_pandas()

weekly_injuries = injuries.loc[
    (injuries["season"] == SEASON)
    & (injuries["week"] == TARGET_WEEK)
].copy()

weekly_injuries["injury_source"] = "weekly_report"

# Reserve designations used by the historical model
INJURY_RESERVE_CODES = {
    "R01", "R04", "R05", "R48"
}

# Roster week is not a weekly snapshot.
# Use all current-season records and their latest status.
current_roster = rosters.loc[
    rosters["season"] == SEASON
].copy()

long_term_injuries = current_roster.loc[
    (current_roster["status"] == "RES")
    & current_roster["status_description_abbr"].isin(
        INJURY_RESERVE_CODES
    )
].copy()

long_term_injuries = long_term_injuries[
    [
        "season",
        "team",
        "week",
        "gsis_id",
        "position",
        "full_name"
    ]
].copy()

long_term_injuries["week"] = TARGET_WEEK
long_term_injuries["report_primary_injury"] = "Long-term reserve"
long_term_injuries["report_secondary_injury"] = np.nan
long_term_injuries["report_status"] = "Out"
long_term_injuries["practice_primary_injury"] = np.nan
long_term_injuries["practice_secondary_injury"] = np.nan
long_term_injuries["practice_status"] = np.nan
long_term_injuries["injury_source"] = "long_term_reserve"

# Combine and deduplicate player availability records
injuries = pd.concat(
    [weekly_injuries, long_term_injuries],
    ignore_index=True,
    sort=False
)

injuries["source_priority"] = (
    injuries["injury_source"]
    .map({
        "weekly_report": 1,
        "long_term_reserve": 2
    })
    .fillna(0)
)

injuries = (
    injuries
    .sort_values("source_priority", ascending=False)
    .drop_duplicates(
        subset=["team", "gsis_id"],
        keep="first"
    )
    .drop(columns="source_priority")
    .reset_index(drop=True)
)

print("Week injury-report rows:", len(weekly_injuries))
print("Long-term reserve injuries:", len(long_term_injuries))
print("Combined unavailable/injured players:", len(injuries))

print("\nSources:")
print(injuries["injury_source"].value_counts(dropna=False))

print("\nTeams represented:", injuries["team"].nunique())

assert long_term_injuries["gsis_id"].notna().all()
assert not long_term_injuries["gsis_id"].duplicated().any()

print("\nWeek 9 reserve-player intake validated.")


Week injury-report rows: 0
Long-term reserve injuries: 338
Combined unavailable/injured players: 338

Sources:
injury_source
long_term_reserve    338
Name: count, dtype: int64

Teams represented: 32

Week 9 reserve-player intake validated.


## Keep the Latest Depth Chart

Depth chart rank is used as a player specific role proxy.

A starter matters more than a reserve, but the model deliberately does not assume the starter's full value disappears when he is limited or unavailable.


In [4]:
if "dt" in depth.columns:
    depth["dt"] = pd.to_datetime(depth["dt"], errors="coerce")
    latest_dt = depth["dt"].max()
    latest_depth = depth[depth["dt"] == latest_dt].copy()
else:
    latest_depth = depth.copy()
    latest_dt = None

depth_cols = [
    c for c in [
        "team", "player_name", "gsis_id", "pos_grp",
        "pos_name", "pos_abb", "pos_slot", "pos_rank"
    ]
    if c in latest_depth.columns
]

latest_depth = latest_depth[depth_cols].copy()

print("Latest depth-chart date:", latest_dt)
print("Depth-chart rows:", len(latest_depth))


Latest depth-chart date: 2026-11-24 07:02:17+00:00
Depth-chart rows: 2383


## Availability / Severity Translation

The injury type itself is retained for diagnostics, but we do not assign made up point values to labels such as ankle, hamstring, shoulder, or knee.

Instead, severity enters through the team's official participation and game status information.

The values below represent expected lost availability, not expected lost team value:

- Out / IR-like designation: 100%
- Doubtful: 80%
- Questionable: 35%
- Full practice / no game designation: 0%
- Limited practice without a designation: 10%
- Did not practice without a designation: 25%

These are deliberately conservative and can later be replaced with probabilities estimated from historical injury report to snap outcomes.


In [5]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()

def expected_lost_availability(row):
    report_status = str(row.get("report_status", "")).lower()
    practice_status = str(row.get("practice_status", "")).lower()

    if "out" in report_status or "reserve" in report_status or "ir" in report_status:
        return 1.00

    if "doubtful" in report_status:
        return 0.80

    if "questionable" in report_status:
        return 0.35

    if "did not participate" in practice_status:
        return 0.25

    if "limited participation" in practice_status:
        return 0.10

    if "full participation" in practice_status:
        return 0.00

    return 0.00

if injuries.empty:
    injuries["expected_lost_availability"] = pd.Series(
        dtype="float64"
    )
else:
    injuries["expected_lost_availability"] = injuries.apply(
        expected_lost_availability,
        axis=1
    )


## Match Injured Players to Their Current Role

GSIS ID is preferred because names can differ across sources. Name matching is used only as a fallback.


In [6]:
injury_detail = injuries.copy()

if "pos_abb" not in injury_detail.columns:
    injury_detail["pos_abb"] = injury_detail.get(
        "position",
        pd.Series("", index=injury_detail.index)
    )

if "pos_rank" not in injury_detail.columns:
    injury_detail["pos_rank"] = np.nan

if len(injury_detail):
    if (
        "gsis_id" in latest_depth.columns
        and "gsis_id" in injury_detail.columns
    ):
        depth_by_id = (
            latest_depth
            .dropna(subset=["gsis_id"])
            .sort_values("pos_rank")
            .drop_duplicates("gsis_id")
        )

        depth_columns = [
            c for c in [
                "gsis_id", "pos_grp", "pos_abb",
                "pos_slot", "pos_rank"
            ]
            if c in depth_by_id.columns
        ]

        depth_lookup = depth_by_id[depth_columns].rename(
            columns={
                c: f"depth_{c}"
                for c in depth_columns
                if c != "gsis_id"
            }
        )

        injury_detail = injury_detail.merge(
            depth_lookup,
            on="gsis_id",
            how="left"
        )

        if "depth_pos_abb" in injury_detail.columns:
            injury_detail["pos_abb"] = (
                injury_detail["depth_pos_abb"]
                .combine_first(injury_detail["pos_abb"])
            )

        if "depth_pos_rank" in injury_detail.columns:
            injury_detail["pos_rank"] = (
                injury_detail["depth_pos_rank"]
                .combine_first(injury_detail["pos_rank"])
            )

if "position" in injury_detail.columns:
    injury_detail["pos_abb"] = (
        injury_detail["pos_abb"]
        .replace("", np.nan)
        .fillna(injury_detail["position"])
        .fillna("")
    )

injury_detail["pos_rank"] = (
    pd.to_numeric(
        injury_detail["pos_rank"],
        errors="coerce"
    )
    .fillna(3)
)

print("Injury detail rows:", len(injury_detail))
print("Position column available:", "pos_abb" in injury_detail.columns)

Injury detail rows: 338
Position column available: True


## Conservative Player Importance

This is where the "next man up" principle enters.

Depth chart role multipliers:

- Rank 1: 1.00
- Rank 2: 0.45
- Rank 3+: 0.20

The maximum effect is also position specific. Quarterback has the largest possible impact, while every non QB position is capped well below that.

These are caps, not automatic deductions. The final deduction is:

`availability loss × role multiplier × position cap × replacement discount`

The replacement discount is 0.60, meaning the model assumes a meaningful share of the player's role can be replaced rather than treating his absence as a complete loss.


In [7]:
# TEMPORARY DIAGNOSTIC — DELETE AFTER REVIEW

print("Injury detail rows:", len(injury_detail))
print("Injury detail shape:", injury_detail.shape)
print("Duplicate columns:", injury_detail.columns[
    injury_detail.columns.duplicated()
].tolist())

Injury detail rows: 338
Injury detail shape: (338, 24)
Duplicate columns: []


In [8]:
depth_history = depth.copy()

depth_history["dt"] = pd.to_datetime(
    depth_history["dt"],
    errors="coerce",
    utc=True
)

depth_history["depth_date"] = depth_history["dt"].dt.date

depth_daily = (
    depth_history
    .sort_values("dt")
    .drop_duplicates(
        subset=["depth_date", "team", "gsis_id"],
        keep="last"
    )
    .copy()
)

depth_daily["pos_rank"] = pd.to_numeric(
    depth_daily["pos_rank"],
    errors="coerce"
)


def get_baseline_pos_rank(row):
    """
    Estimate the player's healthy role from depth-chart history.

    Start from the current rank. Look backward through the player's
    depth chart history and identify the most recent sustained better
    role.

    A role must appear on at least 3 separate days so a one-day depth
    chart fluctuation does not redefine the player's baseline.
    """

    player_history = depth_daily[
        (depth_daily["team"] == row["team"])
        & (depth_daily["gsis_id"] == row["gsis_id"])
    ].copy()

    if player_history.empty:
        return row["pos_rank"]

    player_history = (
        player_history
        .dropna(subset=["pos_rank"])
        .sort_values("dt")
    )

    if player_history.empty:
        return row["pos_rank"]

    current_rank = row["pos_rank"]
    latest_date = player_history["dt"].max()

    recent_history = player_history[
        player_history["dt"]
        >= latest_date - pd.Timedelta(days=45)
    ].copy()

    if recent_history.empty:
        return current_rank

    rank_counts = (
        recent_history
        .groupby("pos_rank")["depth_date"]
        .nunique()
    )

    established_ranks = rank_counts[
        rank_counts >= 3
    ].index.tolist()

    if not established_ranks:
        return current_rank

    better_ranks = [
        rank
        for rank in established_ranks
        if rank < current_rank
    ]

    if better_ranks:
        return float(min(better_ranks))

    return float(current_rank)


if injury_detail.empty:
    injury_detail["baseline_pos_rank"] = pd.Series(
        dtype="float64"
    )
else:
    injury_detail["baseline_pos_rank"] = injury_detail.apply(
        get_baseline_pos_rank,
        axis=1
    )

print("QB ROLE CHECK")

if injury_detail.empty:
    print("No injury-detail records available for QB role check.")
else:
    qb_role_check = injury_detail.loc[
        injury_detail["position"].eq("QB")
    ].copy()

    qb_role_check = injury_detail[
    [
        "team",
        "full_name",
        "position",
        "pos_rank",
        "baseline_pos_rank",
        "expected_lost_availability"
    ]
].copy()

display(qb_role_check)

if injury_detail.empty:
    print("No QB injury records available for Week 6.")
else:
    qb_role_check = injury_detail.loc[
        injury_detail["position"].eq("QB")
    ].copy()

    if qb_role_check.empty:
        print("No QB injuries to display.")
    else:
        display(
            qb_role_check.sort_values(
                ["team", "baseline_pos_rank", "pos_rank"]
            )
        )


QB ROLE CHECK


,team,full_name,position,pos_rank,baseline_pos_rank,expected_lost_availability
0,LA,Davante Adams,WR,5.0,2.0,1.0
1,CLE,Michael Burton,RB,1.0,1.0,1.0
2,HOU,Mario Edwards,DL,3.0,3.0,1.0
3,WAS,Laremy Tunsil,OL,3.0,3.0,1.0
4,NYG,Roy Robertson-Harris,DL,3.0,3.0,1.0
...,...,...,...,...,...,...
333,ARI,Kaleb Proctor,DL,4.0,3.0,1.0
334,SF,Gracen Halton,DL,4.0,1.0,1.0
335,NYJ,Connor Hulstein,TE,3.0,3.0,1.0
336,LA,Daniel Rickert,LB,3.0,3.0,1.0


,season,season_type,game_type,team,week,gsis_id,position,full_name,first_name,last_name,...,practice_status,injury_source,expected_lost_availability,pos_abb,pos_rank,depth_pos_grp,depth_pos_abb,depth_pos_slot,depth_pos_rank,baseline_pos_rank
45,2026,NaN,NaN,ARI,12,00-0035289,QB,Gardner Minshew,NaN,NaN,...,NaN,long_term_reserve,1.0,QB,4.0,3WR 1TE,QB,9.0,4.0,1.0
33,2026,NaN,NaN,BAL,12,00-0034796,QB,Lamar Jackson,NaN,NaN,...,NaN,long_term_reserve,1.0,QB,4.0,3WR 1TE,QB,9.0,4.0,1.0
58,2026,NaN,NaN,BAL,12,00-0035993,QB,Tyler Huntley,NaN,NaN,...,NaN,long_term_reserve,1.0,QB,5.0,3WR 1TE,QB,9.0,5.0,1.0
120,2026,NaN,NaN,BAL,12,00-0037327,QB,Skylar Thompson,NaN,NaN,...,NaN,long_term_reserve,1.0,QB,6.0,3WR 1TE,QB,9.0,6.0,4.0
253,2026,NaN,NaN,HOU,12,00-0040211,QB,Graham Mertz,NaN,NaN,...,NaN,long_term_reserve,1.0,QB,3.0,3WR 1TE,QB,9.0,3.0,3.0
201,2026,NaN,NaN,JAX,12,00-0039534,QB,Carter Bradley,NaN,NaN,...,NaN,long_term_reserve,1.0,QB,3.0,NaN,NaN,NaN,NaN,3.0
119,2026,NaN,NaN,KC,12,00-0037324,QB,Chris Oladokun,NaN,NaN,...,NaN,long_term_reserve,1.0,QB,3.0,NaN,NaN,NaN,NaN,3.0
321,2026,NaN,NaN,LA,12,00-0041414,QB,Matthew Caldwell,NaN,NaN,...,NaN,long_term_reserve,1.0,QB,4.0,3WR 1TE,QB,9.0,4.0,4.0
97,2026,NaN,NaN,LAC,12,00-0037012,QB,Trey Lance,NaN,NaN,...,NaN,long_term_reserve,1.0,QB,3.0,3WR 1TE,QB,9.0,3.0,3.0
146,2026,NaN,NaN,MIA,12,00-0038128,QB,Malik Willis,NaN,NaN,...,NaN,long_term_reserve,1.0,QB,3.0,3WR 1TE,QB,9.0,3.0,1.0


In [9]:
snap_counts = nfl.load_snap_counts(
    [SEASON - 1, SEASON]
).to_pandas()

snap_history = snap_counts[
    (
        (snap_counts["season"] == SEASON - 1)
        |
        (
            (snap_counts["season"] == SEASON)
            & (snap_counts["week"] < TARGET_WEEK)
        )
    )
].copy()

OFFENSIVE_POSITIONS = {
    "QB", "RB", "FB", "WR", "TE",
    "OL", "T", "OT", "G", "OG", "C"
}

DEFENSIVE_POSITIONS = {
    "DE", "EDGE", "DT", "NT",
    "LB", "OLB", "ILB", "MLB",
    "CB", "DB", "S", "FS", "SS"
}


def get_relevant_snap_pct(row):

    position = normalize_text(
        row["position"]
    ).upper()

    if position in OFFENSIVE_POSITIONS:
        return pd.to_numeric(
            row["offense_pct"],
            errors="coerce"
        )

    if position in DEFENSIVE_POSITIONS:
        return pd.to_numeric(
            row["defense_pct"],
            errors="coerce"
        )

    if position in {"K", "P"}:
        return pd.to_numeric(
            row["st_pct"],
            errors="coerce"
        )

    return np.nan


snap_history["relevant_snap_pct"] = (
    snap_history.apply(
        get_relevant_snap_pct,
        axis=1
    )
)

def healthy_usage_estimate(series):

    values = (
        pd.to_numeric(series, errors="coerce")
        .dropna()
        .sort_values(ascending=False)
        .head(5)
    )

    if values.empty:
        return np.nan

    return values.median()


player_snap_usage = (
    snap_history
    .groupby(
        ["player"],
        as_index=False
    )
    .agg(
        snap_games_available=(
            "game_id",
            "nunique"
        ),
        healthy_snap_pct=(
            "relevant_snap_pct",
            healthy_usage_estimate
        )
    )
)

def usage_multiplier(snap_pct):

    if pd.isna(snap_pct):
        return 1.00

    if snap_pct >= 0.90:
        return 1.00

    if snap_pct >= 0.75:
        return 0.90

    if snap_pct >= 0.60:
        return 0.80

    if snap_pct >= 0.40:
        return 0.65

    if snap_pct >= 0.20:
        return 0.45

    return 0.25


player_snap_usage["usage_multiplier"] = (
    player_snap_usage["healthy_snap_pct"]
    .apply(usage_multiplier)
)

injury_detail = injury_detail.merge(
    player_snap_usage,
    left_on="full_name",
    right_on="player",
    how="left"
)

qb_mask = (
    injury_detail["pos_abb"]
    .apply(normalize_text)
    .str.upper()
    .eq("QB")
)

injury_detail.loc[
    qb_mask,
    "usage_multiplier"
] = 1.00

injury_detail["usage_multiplier"] = (
    injury_detail["usage_multiplier"]
    .fillna(1.00)
)

print("HEALTHY USAGE CHECK")

usage_check_cols = [
    "team",
    "full_name",
    "position",
    "pos_rank",
    "baseline_pos_rank",
    "snap_games_available",
    "healthy_snap_pct",
    "usage_multiplier",
    "expected_lost_availability"
]

display(
    injury_detail[
        usage_check_cols
    ]
    .sort_values(
        [
            "expected_lost_availability",
            "healthy_snap_pct"
        ],
        ascending=[False, False]
    )
    .head(40)
    .round(3)
)

HEALTHY USAGE CHECK


,team,full_name,position,pos_rank,baseline_pos_rank,snap_games_available,healthy_snap_pct,usage_multiplier,expected_lost_availability
3,WAS,Laremy Tunsil,OL,3.0,3.0,14.0,1.0,1.0,1.0
5,JAX,Eric Murray,DB,3.0,1.0,19.0,1.0,1.0,1.0
9,CAR,Taylor Moton,OL,3.0,1.0,19.0,1.0,1.0,1.0
14,BAL,Ethan Pocic,OL,4.0,3.0,14.0,1.0,1.0,1.0
28,NYG,Tremaine Edmunds,LB,3.0,1.0,21.0,1.0,1.0,1.0
32,LAC,Derwin James,DB,3.0,1.0,20.0,1.0,1.0,1.0
33,BAL,Lamar Jackson,QB,4.0,1.0,18.0,1.0,1.0,1.0
41,JAX,Patrick Mekari,OL,3.0,3.0,15.0,1.0,1.0,1.0
43,HOU,Azeez Al-Shaair,LB,2.0,1.0,25.0,1.0,1.0,1.0
46,ARI,Andrew Wingard,DB,3.0,1.0,24.0,1.0,1.0,1.0


In [10]:
ROLE_MULTIPLIER = {
    1: 1.00,
    2: 0.45
}

POSITION_POINT_CAP = {
    "QB": 4.00,
    "OL": 0.70,
    "T": 0.70,
    "OT": 0.70,
    "G": 0.55,
    "OG": 0.55,
    "C": 0.55,
    "WR": 0.65,
    "TE": 0.45,
    "RB": 0.35,
    "FB": 0.20,
    "DE": 0.60,
    "EDGE": 0.60,
    "DT": 0.45,
    "NT": 0.40,
    "LB": 0.45,
    "OLB": 0.50,
    "ILB": 0.40,
    "MLB": 0.40,
    "CB": 0.60,
    "DB": 0.50,
    "S": 0.50,
    "FS": 0.50,
    "SS": 0.50,
    "K": 0.20,
    "P": 0.10
}

REPLACEMENT_DISCOUNT = 0.60


def role_multiplier(rank, position):
    """
    Convert healthy baseline depth-chart role into a player
    importance multiplier.

    QB uses a steeper depth curve because only one QB normally
    plays meaningful offensive snaps.
    """

    if pd.isna(rank):
        return 0.20

    rank = int(rank)
    position = normalize_text(position).upper()

    if position == "QB":
        if rank <= 1:
            return 1.00
        if rank == 2:
            return 0.10
        if rank == 3:
            return 0.02
        return 0.00

    if rank <= 1:
        return 1.00
    if rank <= 2:
        return 0.45
    return 0.20


def position_cap(position):
    return POSITION_POINT_CAP.get(
        normalize_text(position).upper(),
        0.25
    )


if len(injury_detail):
    injury_detail["role_multiplier"] = injury_detail.apply(
        lambda row: role_multiplier(
            row["baseline_pos_rank"],
            row["pos_abb"]
        ),
        axis=1
    )

    injury_detail["position_point_cap"] = (
        injury_detail["pos_abb"].apply(position_cap)
    )

    injury_detail["raw_injury_point_cost"] = (
        injury_detail["expected_lost_availability"]
        * injury_detail["role_multiplier"]
        * injury_detail["usage_multiplier"]
        * injury_detail["position_point_cap"]
    )

    injury_detail["injury_point_cost"] = (
        injury_detail["raw_injury_point_cost"]
        * REPLACEMENT_DISCOUNT
    )

## Team Level Injury Adjustment

Multiple injuries can accumulate, but the total team adjustment is capped.

This prevents a long injury report from mechanically destroying a team's rating and reflects the fact that NFL teams plan for depth.

A negative value means the current injury situation lowers the team's weekly rating.


In [11]:

# Aggregate weekly reports and long-term reserve injuries.
# Review results before saving.

MAX_TEAM_INJURY_PENALTY = 4.5

verified_injury_detail = injury_detail.loc[
    injury_detail["injury_source"].isin(
        ["weekly_report", "long_term_reserve"]
    )
].copy()

assert not verified_injury_detail["gsis_id"].duplicated().any(), (
    "Duplicate player IDs detected."
)

assert verified_injury_detail["injury_point_cost"].notna().all(), (
    "Missing player injury costs."
)

team_injuries = (
    verified_injury_detail
    .groupby("team", as_index=False)
    .agg(
        injury_report_players=("gsis_id", "count"),
        injury_point_cost=("injury_point_cost", "sum")
    )
)

team_injuries["injury_adjustment"] = -np.minimum(
    team_injuries["injury_point_cost"],
    MAX_TEAM_INJURY_PENALTY
)

injury_adjusted = team_strength.merge(
    team_injuries,
    on="team",
    how="left",
    validate="one_to_one"
)

cols = [
    "injury_report_players",
    "injury_point_cost",
    "injury_adjustment"
]

injury_adjusted[cols] = injury_adjusted[cols].fillna(0)

injury_adjusted["pre_injury_weekly_team_strength"] = (
    injury_adjusted["weekly_team_strength"]
)

injury_adjusted["injury_adjusted_team_strength"] = (
    injury_adjusted["pre_injury_weekly_team_strength"]
    + injury_adjusted["injury_adjustment"]
)

print("Players included:", len(verified_injury_detail))
print("Total player injury cost:",
      round(verified_injury_detail["injury_point_cost"].sum(), 3))
print("Teams receiving penalties:",
      (injury_adjusted["injury_adjustment"] < 0).sum())
print("Total team adjustment:",
      round(injury_adjusted["injury_adjustment"].sum(), 3))

display(
    injury_adjusted[
        [
            "team",
            "injury_report_players",
            "injury_point_cost",
            "injury_adjustment",
            "injury_adjusted_team_strength"
        ]
    ].sort_values("injury_adjustment")
)


Players included: 338
Total player injury cost: 31.568
Teams receiving penalties: 32
Total team adjustment: -30.775


,team,injury_report_players,injury_point_cost,injury_adjustment,injury_adjusted_team_strength
19,BAL,11,5.292900,-4.500000,-7.049361
30,ARI,16,3.479775,-3.479775,-10.768827
28,MIA,10,2.928675,-2.928675,-8.035464
23,PHI,7,2.903475,-2.903475,-6.341980
4,DET,14,1.256475,-1.256475,4.206555
17,WAS,11,1.117800,-1.117800,-1.870403
3,LA,14,1.055775,-1.055775,4.926084
9,CHI,16,0.969300,-0.969300,2.470995
5,JAX,12,0.848250,-0.848250,4.592927
31,TEN,12,0.815925,-0.815925,-9.104252


## Diagnostics

This is an important review point. Injury adjustments should generally be small.

A major quarterback absence can matter substantially, but ordinary questionable/limited players should move the model only slightly.


In [12]:
print("TEAM INJURY ADJUSTMENTS")
display(
    injury_adjusted[
        [
            "team",
            "pre_injury_weekly_team_strength",
            "injury_report_players",
            "injury_adjustment",
            "injury_adjusted_team_strength"
        ]
    ]
    .sort_values("injury_adjustment")
    .round(3)
)

if len(injury_detail):
    print("\nLARGEST INDIVIDUAL INJURY EFFECTS")
    
    detail_cols = [
        c for c in [
            "team",
            "full_name",
            "position",
            "report_primary_injury",
            "report_status",
            "practice_status",
            "pos_rank",
            "baseline_pos_rank",
            "role_multiplier",
            "expected_lost_availability",
            "injury_point_cost"
        ]
        if c in injury_detail.columns
    ]

    display(
        injury_detail[detail_cols]
        .sort_values(
            "injury_point_cost",
            ascending=False
        )
        .head(25)
        .round(3)
    )

TEAM INJURY ADJUSTMENTS


,team,pre_injury_weekly_team_strength,injury_report_players,injury_adjustment,injury_adjusted_team_strength
19,BAL,-2.549,11,-4.500,-7.049
30,ARI,-7.289,16,-3.480,-10.769
28,MIA,-5.107,10,-2.929,-8.035
23,PHI,-3.439,7,-2.903,-6.342
4,DET,5.463,14,-1.256,4.207
17,WAS,-0.753,11,-1.118,-1.870
3,LA,5.982,14,-1.056,4.926
9,CHI,3.440,16,-0.969,2.471
5,JAX,5.441,12,-0.848,4.593
31,TEN,-8.288,12,-0.816,-9.104



LARGEST INDIVIDUAL INJURY EFFECTS


,team,full_name,position,report_primary_injury,report_status,practice_status,pos_rank,baseline_pos_rank,role_multiplier,expected_lost_availability,injury_point_cost
33,BAL,Lamar Jackson,QB,Long-term reserve,Out,NaN,4.0,1.0,1.00,1.0,2.400
146,MIA,Malik Willis,QB,Long-term reserve,Out,NaN,3.0,1.0,1.00,1.0,2.400
45,ARI,Gardner Minshew,QB,Long-term reserve,Out,NaN,4.0,1.0,1.00,1.0,2.400
72,PHI,Jalen Hurts,QB,Long-term reserve,Out,NaN,4.0,1.0,1.00,1.0,2.400
58,BAL,Tyler Huntley,QB,Long-term reserve,Out,NaN,5.0,1.0,1.00,1.0,2.400
49,WAS,Terry McLaurin,WR,Long-term reserve,Out,NaN,7.0,1.0,1.00,1.0,0.351
116,DET,Cade Mays,OL,Long-term reserve,Out,NaN,3.0,1.0,1.00,1.0,0.330
46,ARI,Andrew Wingard,DB,Long-term reserve,Out,NaN,3.0,1.0,1.00,1.0,0.300
32,LAC,Derwin James,DB,Long-term reserve,Out,NaN,3.0,1.0,1.00,1.0,0.300
5,JAX,Eric Murray,DB,Long-term reserve,Out,NaN,3.0,1.0,1.00,1.0,0.300


## Data Availability Warning

The injury layer is only as current as the official report.

If Week 2 reports are incomplete today, that is expected. Rerun this notebook later in the week before locking the final projection board.


In [13]:
if len(injuries) == 0:
    print(
        "WARNING: No official Week "
        f"{TARGET_WEEK} injury rows are available yet. "
        "Current injury adjustment is therefore zero. "
        "Rerun this notebook when reports are published."
    )
elif injuries["report_status"].isna().mean() > 0.75:
    print(
        "WARNING: Most players do not yet have final game-status "
        "designations. Treat these injury adjustments as preliminary "
        "and rerun closer to kickoff."
    )
else:
    print("Injury report contains substantial game status information.")


Injury report contains substantial game status information.


In [14]:
WEEKLY_DATA_DIR.mkdir(parents=True, exist_ok=True)

injury_adjusted.to_parquet(OUTPUT_PATH, index=False)
injury_detail.to_parquet(INJURY_DETAIL_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Saved:", INJURY_DETAIL_PATH)


Saved: ..\..\data\processed\weekly\week_12_injury_adjusted_team_strength.parquet
Saved: ..\..\data\processed\weekly\week_12_injury_adjustments.parquet
